# DATASCI 281 — Group 2 Final Project
## Dataset Exploration: WOD-E2E Scenario Classification

**Before running:** Make sure you have:
1. Activated the conda env: `conda activate 281-s2-group2`
2. Downloaded the val shard and labels to `../data/` (see README)

In [ ]:
import json
import os
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image

DATA_DIR = Path('../data')
LABELS_FILE = DATA_DIR / 'val_sequence_name_to_scenario_cluster.json'
TFRECORD_FILE = DATA_DIR / 'val_202504211843.tfrecord-00000-of-00093'

print('TF version:', tf.__version__)
print('Data dir exists:', DATA_DIR.exists())
print('Labels file exists:', LABELS_FILE.exists())
print('TFRecord exists:', TFRECORD_FILE.exists())

## 1. Dataset Overview

In [ ]:
with open(LABELS_FILE) as f:
    labels = json.load(f)

clusters = [v['scenario_cluster'] for v in labels.values()]
counts = Counter(clusters)

print(f'Total sequences in val set: {len(labels)}')
print(f'Classes: {sorted(counts.keys())}')
print()
for cls, count in sorted(counts.items(), key=lambda x: -x[1]):
    print(f'  {cls:<30} {count}')

In [ ]:
# Class distribution plot
fig, ax = plt.subplots(figsize=(10, 4))
sorted_counts = sorted(counts.items(), key=lambda x: -x[1])
ax.bar([c[0] for c in sorted_counts], [c[1] for c in sorted_counts])
ax.set_xticklabels([c[0] for c in sorted_counts], rotation=45, ha='right')
ax.set_ylabel('Number of sequences')
ax.set_title('Val set class distribution')
plt.tight_layout()
plt.show()

## 2. Extract and Visualize Sample Frames

**Note:** We are currently using naive JPEG byte-scanning for extraction.
The proper approach is via `StandardE2E`'s `PanoImageAdapter` — see README for setup.
This requires NumPy < 2 to work with TensorFlow.

In [ ]:
def extract_first_jpeg(raw_bytes):
    """Extract first JPEG from a raw tfrecord bytes object."""
    idx = raw_bytes.find(b'\xff\xd8\xff')
    if idx == -1:
        return None
    end = raw_bytes.find(b'\xff\xd9', idx) + 2
    return raw_bytes[idx:end]

def get_seq_id(raw_bytes):
    """Extract sequence ID from tfrecord key (format: {seq_id}-{frame_num})."""
    example = tf.train.Example()
    example.ParseFromString(raw_bytes)
    keys = list(example.features.feature.keys())
    if not keys:
        return None
    return keys[0].split('-')[0]

# Extract one frame per class
samples = {}  # cluster -> PIL Image
dataset = tf.data.TFRecordDataset(str(TFRECORD_FILE))

for raw in dataset:
    b = raw.numpy()
    seq_id = get_seq_id(b)
    if seq_id not in labels:
        continue
    cluster = labels[seq_id]['scenario_cluster']
    if cluster in samples:
        continue
    jpeg = extract_first_jpeg(b)
    if jpeg:
        import io
        samples[cluster] = Image.open(io.BytesIO(jpeg))
    if len(samples) >= 10:
        break

print(f'Extracted {len(samples)} sample frames')

In [ ]:
# Visualize one frame per class
n = len(samples)
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for ax, (cluster, img) in zip(axes.flat, sorted(samples.items())):
    ax.imshow(img)
    ax.set_title(cluster, fontsize=9)
    ax.axis('off')
plt.suptitle('One sample frame per scenario class (naive extraction)', y=1.02)
plt.tight_layout()
plt.show()

## 3. TODO: Proper Extraction via StandardE2E

Once the environment is set up with `numpy<2`, replace the naive extraction above with:

```python
from standard_e2e.caching.adapters import PanoImageAdapter
from standard_e2e.caching.src_datasets.waymo_e2e import WaymoE2EDatasetProcessor, WaymoE2EDatasetConverter
```

See the [StandardE2E docs](https://standarde2e.readthedocs.io/en/latest/) and `examples/dataset_preprocessing.py` in that repo.

## 4. Feature Extraction (Stub)

Planned features:
- HOG (simple feature)
- 1/f spectral slope (simple feature)
- Fourier features (simple feature)
- CLIP / DINOv2 embeddings (complex pretrained feature)

In [ ]:
# TODO: implement feature extractors
# from skimage.feature import hog
# from skimage.color import rgb2gray
# import numpy as np

# def extract_hog(img):
#     gray = rgb2gray(np.array(img))
#     features = hog(gray, orientations=9, pixels_per_cell=(8,8), cells_per_block=(2,2))
#     return features

print('Feature extraction stubs — implement me!')

## 5. Classification (Stub)

Planned classifiers:
- SVM
- Random Forest
- (stretch) fine-tuned linear probe on CLIP/DINOv2 embeddings

In [ ]:
# TODO: implement classifiers
# from sklearn.svm import SVC
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.model_selection import GridSearchCV

print('Classification stubs — implement me!')